# Result · In-loop statistics of every patch loop

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** Fig. 2b, the pair-side ablation of Sec. III (in-loop ASR by round 3) and the replay-buffer sizes of Sec. IV.

Per round: pairs found, replay-buffer size, and the success rate of the round's attacker against R<sub>k-1</sub>
under both references, for the main loops, the ablations and the multi-space loops. Pair counts are training records
(from the released `done.json`); the attack success rates are recomputed (12 jobs, 0.5–1 h each).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['hnp', 'ablation', 'multispace']]
print(len(KEYS), "runs:", ", ".join(KEYS))

12 runs: hnp_s0, hnp_s1, hnp_s2, abl_tau0.1_s0, abl_margin0.1_s0, abl_headonly_s0, abl_tau0.5_s0, abl_margin1.0_s0, abl_prefinput_s0, ms_s0, ms_s1, ms_s2


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    12
Name: count, dtype: int64

In [6]:
lr = pd.read_csv(OUT / "loop_rounds_both_refs.csv")
lr.pivot_table(index=["run", "space"], columns="round", values="asr_ref_recon").round(3)

,round,1,2,3
run,space,,,
abl_headonly_s0,encodec,0.256,0.318,0.096
abl_margin0.1_s0,encodec,0.256,0.054,0.005
abl_margin1.0_s0,encodec,0.256,0.082,0.000
abl_prefinput_s0,encodec,0.256,0.193,0.172
abl_tau0.1_s0,encodec,0.256,0.012,0.000
abl_tau0.5_s0,encodec,0.256,0.047,0.035
hnp_s0,encodec,0.256,0.094,0.010
hnp_s1,encodec,0.220,0.024,0.080
hnp_s2,encodec,0.373,0.000,0.000


In [7]:
display(Markdown((OUT / "loop_rounds.md").read_text()))

| run              |   round |   n_pairs_round |   n_pairs_buffer |   train_asr |   val_asr |   val_mean_gain |   val_ceiling_rate |   train_asr_encodec |   val_asr_encodec |   train_asr_dac |   val_asr_dac |   train_asr_pgd |   val_asr_pgd |
|:-----------------|--------:|----------------:|-----------------:|------------:|----------:|----------------:|-------------------:|--------------------:|------------------:|----------------:|--------------:|----------------:|--------------:|
| hnp_s0           |       1 |            4062 |             4062 |       0.209 |     0.185 |           0.098 |              0.230 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s0           |       2 |            2284 |             6346 |       0.116 |     0.060 |           0.019 |              0.138 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s0           |       3 |            2068 |             8414 |       0.102 |     0.060 |           0.178 |              0.273 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s1           |       1 |            4405 |             4405 |       0.227 |     0.230 |           0.115 |              0.250 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s1           |       2 |             894 |             5299 |       0.045 |     0.028 |           0.104 |              0.200 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s1           |       3 |            3557 |             8856 |       0.180 |     0.135 |           0.147 |              0.250 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s2           |       1 |            5684 |             5684 |       0.290 |     0.310 |           0.176 |              0.315 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s2           |       2 |             107 |             5791 |       0.005 |     0.000 |          -0.467 |              0.005 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| hnp_s2           |       3 |              20 |             5811 |       0.001 |     0.000 |          -0.878 |              0.000 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_tau0.1_s0    |       1 |            8163 |             8163 |       0.418 |     0.347 |           0.098 |              0.430 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_tau0.1_s0    |       2 |            3062 |            11225 |       0.154 |     0.090 |          -0.258 |              0.117 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_tau0.1_s0    |       3 |               0 |            11225 |       0.000 |     0.003 |          -1.608 |              0.003 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_margin0.1_s0 |       1 |            4062 |             4062 |       0.209 |     0.185 |           0.098 |              0.230 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_margin0.1_s0 |       2 |            1381 |             5443 |       0.069 |     0.043 |          -0.005 |              0.125 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_margin0.1_s0 |       3 |            2123 |             7566 |       0.107 |     0.090 |           0.061 |              0.182 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_headonly_s0  |       1 |            4062 |             4062 |       0.209 |     0.185 |           0.098 |              0.230 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_headonly_s0  |       2 |            3585 |             7647 |       0.180 |     0.128 |           0.059 |              0.205 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_headonly_s0  |       3 |            2796 |            10443 |       0.139 |     0.100 |           0.093 |              0.217 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_tau0.5_s0    |       1 |            1737 |             1737 |       0.090 |     0.058 |           0.098 |              0.072 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_tau0.5_s0    |       2 |             507 |             2244 |       0.027 |     0.018 |           0.030 |              0.062 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_tau0.5_s0    |       3 |             694 |             2938 |       0.037 |     0.030 |           0.032 |              0.075 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_margin1.0_s0 |       1 |            4062 |             4062 |       0.209 |     0.185 |           0.098 |              0.230 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_margin1.0_s0 |       2 |            2291 |             6353 |       0.116 |     0.080 |           0.053 |              0.170 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_margin1.0_s0 |       3 |            1198 |             7551 |       0.060 |     0.040 |          -0.814 |              0.040 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_prefinput_s0 |       1 |            4062 |             4062 |       0.209 |     0.185 |           0.098 |              0.230 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_prefinput_s0 |       2 |            1775 |             5837 |       0.092 |     0.050 |          -0.045 |              0.098 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| abl_prefinput_s0 |       3 |             675 |             6512 |       0.036 |     0.015 |          -0.238 |              0.022 |             nan     |           nan     |         nan     |       nan     |         nan     |       nan     |
| ms_s0            |       1 |            6687 |             6687 |       0.163 |     0.175 |           0.078 |              0.195 |               0.163 |             0.175 |           0.262 |         0.270 |           0.969 |         0.970 |
| ms_s0            |       2 |            3397 |            10084 |       0.112 |     0.098 |           0.385 |              0.482 |               0.112 |             0.098 |           0.006 |         0.000 |           0.865 |         0.840 |
| ms_s0            |       3 |            3925 |            14009 |       0.292 |     0.312 |           0.182 |              0.333 |               0.292 |             0.312 |           0.008 |         0.003 |           0.382 |         0.453 |
| ms_s1            |       1 |            6998 |             6998 |       0.198 |     0.198 |           0.125 |              0.242 |               0.198 |             0.198 |           0.257 |         0.217 |           0.977 |         0.970 |
| ms_s1            |       2 |            6463 |            13461 |       0.292 |     0.185 |           0.396 |              0.540 |               0.292 |             0.185 |           0.161 |         0.120 |           0.792 |         0.790 |
| ms_s1            |       3 |            6422 |            19883 |       0.562 |     0.505 |           0.495 |              0.583 |               0.562 |             0.505 |           0.028 |         0.018 |           0.226 |         0.200 |
| ms_s2            |       1 |            5272 |             5272 |       0.242 |     0.302 |           0.169 |              0.312 |               0.242 |             0.302 |           0.041 |         0.037 |           0.988 |         0.995 |
| ms_s2            |       2 |           10678 |            15950 |       0.556 |     0.525 |           0.693 |              0.733 |               0.556 |             0.525 |           0.293 |         0.305 |           0.875 |         0.900 |
| ms_s2            |       3 |           10215 |            26165 |       0.824 |     0.845 |           0.979 |              0.877 |               0.824 |             0.845 |           0.092 |         0.080 |           0.430 |         0.463 |

In [8]:
repro.compare_csv(OUT / "loop_rounds_both_refs.csv", COMMITTED_OUT / "loop_rounds_both_refs.csv")

round            0.0
asr_ref_input    0.0
asr_ref_recon    0.0
recon_gain       0.0
n_pairs          0.0
Name: max_abs_diff, dtype: float64

In [9]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('loop_', 'buffer_', 'hack_input', 'all_input', 'panel_'))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

82 numbers; max |diff| = 0.0000; identical: 82


,paper,reproduced,abs_diff
all_input_score_mean,3.2,3.2,0.0
buffer_abl_headonly_s0,10443.0,10443.0,0.0
buffer_abl_margin0.1_s0,7566.0,7566.0,0.0
buffer_abl_margin1.0_s0,7551.0,7551.0,0.0
buffer_abl_prefinput_s0,6512.0,6512.0,0.0
buffer_abl_tau0.1_s0,11225.0,11225.0,0.0
buffer_abl_tau0.5_s0,2938.0,2938.0,0.0
buffer_hnp_s0,8414.0,8414.0,0.0
buffer_hnp_s1,8856.0,8856.0,0.0
buffer_hnp_s2,5811.0,5811.0,0.0
